# PVT v2 + MoE — Colab / no-checkout launcher

For a machine with no clone of the repo. Installs the package from git at a
**pinned commit**, then runs exactly the same code as `python train.py`.

If you have a checkout, use `notebooks/v11_train.ipynb` instead — it imports
the working tree, so your edits take effect without reinstalling.

| | |
|---|---|
| Recipes, defaults, ablation ladder | `docs/HPARAMS.md` |
| Model invariants | `docs/ARCHITECTURE.md` |


## 1. Install

`COMMIT` pins what you are reproducing — always a 40-character sha, never a
branch name, or the notebook silently means something different next month.
Copy it from `git log` (or the run's `results.json` → `identity.git_commit`).

Credentials come from the environment, never from a cell:
`os.environ["HF_TOKEN"]` for pretrained weights, `WANDB_API_KEY` for logging
(or set `use_wandb: False` below). On Colab use the secrets panel.


In [ ]:
REPO   = 'https://github.com/salted-caramel-icecream/Thesis'
COMMIT = '8518af2d348a18e5326116577906ec8428cd7daa'   # 40-char sha, never a branch

assert len(COMMIT) == 40 and all(c in '0123456789abcdef' for c in COMMIT), (
    f'COMMIT must be a full 40-char sha, got {COMMIT!r}')

import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                       f'git+{REPO}@{COMMIT}'])

# Tutel is the default MoE backend and builds a CUDA extension (a few
# minutes, once). Skip it and pass --backend native for a pure-torch run.
import importlib
if importlib.util.find_spec('tutel') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-v', '-U',
                           '--no-build-isolation',
                           'git+https://github.com/microsoft/tutel@main'])

import pvt_moe
print('pvt_moe OK:', pvt_moe.__file__)


## 2. Run

The arm is a command line, not a config file: `--ladder N` picks a
spec-defined architecture row and the budget flags are independent of it.
`docs/HPARAMS.md` §4 lists the rows and the flag crosses.

`--dry-run` resolves and prints the whole config — run name, LR schedule,
effective batch, placements — without building or training anything. Always
look at that before spending compute.


In [ ]:
from pvt_moe.cli import main

ARGS = ['--recipe', 'scratch', '--ladder', '4', '--epochs', '90',
        '--data-dir', '/content/imagenet_arrow',
        '--checkpoint-root', '/content/runs',
        '--dry-run']            # <- drop this line to actually train

main(ARGS)
